<a href="https://colab.research.google.com/github/mutagi/GenAIEngineering-Cohort2/blob/main/GPT_2_Shakespear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Train a GPT-2 Text-Generating Model w/ GPU For Free

by [Max Woolf](http://minimaxir.com)

*Last updated: October 17th, 2021*

Retrain an advanced text generating neural network on any text dataset **for free on a GPU using Collaboratory** using `gpt-2-simple`!

For more about `gpt-2-simple`, you can visit [this GitHub repository](https://github.com/minimaxir/gpt-2-simple). You can also read my [blog post](https://minimaxir.com/2019/09/howto-gpt2/) for more information how to use this notebook!


To get started:

1. Copy this notebook to your Google Drive to keep it and save your changes. (File -> Save a Copy in Drive)
2. Make sure you're running the notebook in Google Chrome.
3. Run the cells below:


In [1]:
!pip install -q gpt-2-simple wandb
import gpt_2_simple as gpt2
from datetime import datetime
from google.colab import files
import wandb

  Preparing metadata (setup.py) ... done


## GPU

Colaboratory uses either a Nvidia T4 GPU or an Nvidia K80 GPU. The T4 is slightly faster than the old K80 for training GPT-2, and has more memory allowing you to train the larger GPT-2 models and generate more text.

You can verify which GPU is active by running the cell below.

In [2]:
!nvidia-smi

Sat Nov 29 15:57:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Downloading GPT-2

If you're retraining a model on new text, you need to download the GPT-2 model first.

There are three released sizes of GPT-2:

* `124M` (default): the "small" model, 500MB on disk.
* `355M`: the "medium" model, 1.5GB on disk.
* `774M`: the "large" model, cannot currently be finetuned with Colaboratory but can be used to generate text from the pretrained model (see later in Notebook)
* `1558M`: the "extra large", true model. Will not work if a K80/P4 GPU is attached to the notebook. (like `774M`, it cannot be finetuned).

Larger models have more knowledge, but take longer to finetune and longer to generate text. You can specify which base model to use by changing `model_name` in the cells below.

The next cell downloads it from Google Cloud Storage and saves it in the Colaboratory VM at `/models/<model_name>`.

This model isn't permanently saved in the Colaboratory VM; you'll have to redownload it if you want to retrain it at a later time.

In [3]:
gpt2.download_gpt2(model_name="124M")

Fetching checkpoint: 1.05Mit [00:00, 3.86Git/s]                                                     
Fetching encoder.json: 1.05Mit [00:01, 656kit/s]
Fetching hparams.json: 1.05Mit [00:00, 5.55Git/s]                                                   
Fetching model.ckpt.data-00000-of-00001: 498Mit [02:13, 3.72Mit/s]
Fetching model.ckpt.index: 1.05Mit [00:00, 4.44Git/s]                                               
Fetching model.ckpt.meta: 1.05Mit [00:01, 845kit/s]
Fetching vocab.bpe: 1.05Mit [00:01, 837kit/s]


## Mounting Google Drive

The best way to get input text to-be-trained into the Colaboratory VM, and to get the trained model *out* of Colaboratory, is to route it through Google Drive *first*.

Running this cell (which will only work in Colaboratory) will mount your personal Google Drive in the VM, which later cells can use to get data in/out. (it will ask for an auth code; that auth is not saved anywhere)

In [4]:
gpt2.mount_gdrive()

MessageError: Error: credential propagation was unsuccessful

## Uploading a Text File to be Trained to Colaboratory

In the Colaboratory Notebook sidebar on the left of the screen, select *Files*. From there you can upload files:

![alt text](https://i.imgur.com/TGcZT4h.png)

Upload **any smaller text file**  (<10 MB) and update the file name in the cell below, then run the cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
file_name = "shakespeare.txt"

If your text file is larger than 10MB, it is recommended to upload that file to Google Drive first, then copy that file from Google Drive to the Colaboratory VM.

In [ ]:
gpt2.copy_file_from_gdrive(file_name)

## Finetune GPT-2

The next cell will start the actual finetuning of GPT-2. It creates a persistent TensorFlow session which stores the training config, then runs the training for the specified number of `steps`. (to have the finetuning run indefinitely, set `steps = -1`)

The model checkpoints will be saved in `/checkpoint/run1` by default. The checkpoints are saved every 500 steps (can be changed) and when the cell is stopped.

The training might time out after 4ish hours; make sure you end training and save the results so you don't lose them!

**IMPORTANT NOTE:** If you want to rerun this cell, **restart the VM first** (Runtime -> Restart Runtime). You will need to rerun imports but not recopy files.

Other optional-but-helpful parameters for `gpt2.finetune`:


*  **`restore_from`**: Set to `fresh` to start training from the base GPT-2, or set to `latest` to restart training from an existing checkpoint.
* **`sample_every`**: Number of steps to print example output
* **`print_every`**: Number of steps to print training progress.
* **`learning_rate`**:  Learning rate for the training. (default `1e-4`, can lower to `1e-5` if you have <1MB input data)
*  **`run_name`**: subfolder within `checkpoint` to save the model. This is useful if you want to work with multiple models (will also need to specify  `run_name` when loading the model)
* **`overwrite`**: Set to `True` if you want to continue finetuning an existing model (w/ `restore_from='latest'`) without creating duplicate copies.

In [ ]:
import os
from google.colab import userdata
import wandb
import tensorflow as tf

# Initialize wandb
# Use Colab secrets for the API key
try:
    wandb_api_key = userdata.get('WANDB_API_KEY')
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.init(project="gpt2-finetune")
except:
    print("Please add your WANDB_API_KEY to Colab secrets for seamless logging.")
    wandb.init(project="gpt2-finetune", mode="disabled")


sess = gpt2.start_tf_sess()

# Define parameters for finetuning
dataset_name = file_name
model_size = '124M'
num_steps = 1000
restore_option = 'fresh'
run_folder = 'run1'
print_frequency = 10
sample_frequency = 200
save_frequency = 500

# Load the model
gpt2.load_gpt2(sess, model_name=model_size)

# Prepare the dataset
dataset = gpt2.load_dataset(sess, dataset_path=dataset_name)

# Finetune the model manually to log loss
loss = gpt2.finetune(sess,
                     dataset=dataset_name, # Pass the dataset_name here
                     model_name=model_size,
                     steps=num_steps,
                     restore_from=restore_option,
                     run_name=run_folder,
                     print_every=print_frequency,
                     sample_every=sample_frequency,
                     save_every=save_frequency,
                     learning_rate=1e-4 # You can adjust this
                     )

# Optional: Log the model checkpoint to wandb
# wandb.save("gpt-2-shakespear/run1/*") # This path might need adjustment

# Optional: End the wandb run
# wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: kingsidharth (gs-test) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading checkpoint models/124M/model.ckpt
Loading dataset...


100%|██████████| 1/1 [00:06<00:00,  6.82s/it]


dataset has 1850440 tokens
Training...
[10 | 24.75] loss=3.30 avg=3.30
[20 | 45.61] loss=2.85 avg=3.07
[30 | 66.86] loss=2.77 avg=2.97
[40 | 88.67] loss=2.85 avg=2.94
[50 | 110.65] loss=2.67 avg=2.89
[60 | 132.50] loss=3.04 avg=2.91
[70 | 154.22] loss=2.91 avg=2.91
[80 | 176.01] loss=2.70 avg=2.88
[90 | 197.85] loss=2.87 avg=2.88
[100 | 219.67] loss=2.55 avg=2.85
[110 | 241.44] loss=2.83 avg=2.85
[120 | 263.21] loss=2.94 avg=2.85
[130 | 284.99] loss=2.95 avg=2.86
[140 | 306.78] loss=2.74 avg=2.85
[150 | 328.57] loss=2.53 avg=2.83
[160 | 350.36] loss=2.60 avg=2.81
[170 | 372.15] loss=2.62 avg=2.80
[180 | 393.94] loss=2.91 avg=2.81
[190 | 415.73] loss=2.58 avg=2.80
[200 | 437.53] loss=2.76 avg=2.79
======== SAMPLE 1 ========
 gate     To bring the Prince
     To his brother's death.
     It is not his fault; but the Prince
     Was brought in haste, and he is dead.
     What wouldst thou do in thine;
     If thou were good, thou mightst have  
     How he, in the time of shame
     Came 

Instructions for updating:
Use standard file APIs to delete files with this prefix.
wandb: WARNING Symlinked 0 file into the W&B run directory, call wandb.save again to sync new files.


After the model is trained, you can copy the checkpoint folder to your own Google Drive.

If you want to download it to your personal computer, it's strongly recommended you copy it there first, then download from Google Drive. The checkpoint folder is copied as a `.rar` compressed file; you can download it and uncompress it locally.

In [ ]:
gpt2.copy_checkpoint_to_gdrive(run_name='run1')

You're done! Feel free to go to the **Generate Text From The Trained Model** section to generate text based on your retrained model.

## Load a Trained Model Checkpoint

Running the next cell will copy the `.rar` checkpoint file from your Google Drive into the Colaboratory VM.

In [ ]:
gpt2.copy_checkpoint_from_gdrive(run_name='run1')

The next cell will allow you to load the retrained model checkpoint + metadata necessary to generate text.

**IMPORTANT NOTE:** If you want to rerun this cell, **restart the VM first** (Runtime -> Restart Runtime). You will need to rerun imports but not recopy files.

In [ ]:
# sess = gpt2.start_tf_sess()
gpt2.load_gpt2(sess, run_name='run1')

ValueError: Variable model/wpe already exists, disallowed. Did you mean to set reuse=True or reuse=tf.AUTO_REUSE in VarScope?

## Generate Text From The Trained Model

After you've trained the model or loaded a retrained model from checkpoint, you can now generate text. `generate` generates a single text from the loaded model.

In [ ]:
gpt2.generate(sess, run_name='run1')

FailedPreconditionError: Graph execution error:

Detected at node 'sample_sequence_2/model/h5/attn/c_proj/add/ReadVariableOp' defined at (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 205, in start
    File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once
    File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_q

If you're creating an API based on your model and need to pass the generated text elsewhere, you can do `text = gpt2.generate(sess, return_as_list=True)[0]`

You can also pass in a `prefix` to the generate function to force the text to start with a given character sequence and generate text from there (good if you add an indicator when the text starts).

You can also generate multiple texts at a time by specifing `nsamples`. Unique to GPT-2, you can pass a `batch_size` to generate multiple samples in parallel, giving a massive speedup (in Colaboratory, set a maximum of 20 for `batch_size`).

Other optional-but-helpful parameters for `gpt2.generate` and friends:

*  **`length`**: Number of tokens to generate (default 1023, the maximum)
* **`temperature`**: The higher the temperature, the crazier the text (default 0.7, recommended to keep between 0.7 and 1.0)
* **`top_k`**: Limits the generated guesses to the top *k* guesses (default 0 which disables the behavior; if the generated output is super crazy, you may want to set `top_k=40`)
* **`top_p`**: Nucleus sampling: limits the generated guesses to a cumulative probability. (gets good results on a dataset with `top_p=0.9`)
* **`truncate`**: Truncates the input text until a given sequence, excluding that sequence (e.g. if `truncate='<|endoftext|>'`, the returned text will include everything before the first `<|endoftext|>`). It may be useful to combine this with a smaller `length` if the input texts are short.
*  **`include_prefix`**: If using `truncate` and `include_prefix=False`, the specified `prefix` will not be included in the returned text.

In [ ]:
gpt2.generate(sess,
              length=250,
              temperature=0.7,
              prefix="Capital of France is",
              nsamples=5,
              batch_size=5
              )

Capital of France is now
    Divide'd from England. For you, sir, your son,
    According to his goods, you shall be a diadem.
    My son shall be a coronet,
    In fair name and in his right honour.
                                                Exit.
  KING HENRY. I think this is not strange. 'Tis not so, I warrant you,
    In all my life.
  BERTRAM. What's your name?
  KING HENRY. My name is Robert.
    Robert is my father's.
  BERTRAM. I pray thee, be sure of it; I have no hands
    That have with a sudden purse or with a desperate purse
    Get the truth of it. I was born in York,
    And now must die in Longford Forest.
  KING HENRY. What is your
Capital of France is divided into five parts:
    The Kentish, the French, the English, and the Danish.
    The French of the Kentish are the King's;
    The English of the Danish are his.
    The English of the Danish are his;
    The English of the French are his;
    The Danish of the Kentish are his;
    The English of the Danish ar

For bulk generation, you can generate a large amount of text to a file and sort out the samples locally on your computer. The next cell will generate a generated text file with a unique timestamp.

You can rerun the cells as many times as you want for even more generated texts!

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/checkpoint_124M.tar'

In [ ]:
import gpt_2_simple as gpt2_new
sess2 = gpt2_new.start_tf_sess()
gpt2_new.load_gpt2(sess2, model_name='124M', reuse=True)

Loading pretrained model models/124M/model.ckpt


NotFoundError: Restoring from checkpoint failed. This is most likely due to a Variable name or other graph key that is missing from the checkpoint. Please ensure that you have not altered the graph expected based on the checkpoint. Original error:

Graph execution error:

Detected at node 'save_5/RestoreV2' defined at (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 205, in start
    File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    File "/usr/lib/python3.12/asyncio/base_events.py", line 19

In [ ]:
gpt2_new.download_gpt2(model_name='124M')

In [ ]:
gen_file = 'gpt2_gentext_{:%Y%m%d_%H%M%S}.txt'.format(datetime.utcnow())

gpt2.generate_to_file(sess,
                      destination_path=gen_file,
                      length=500,
                      temperature=0.7,
                      nsamples=100,
                      batch_size=20
                      )

/tmp/ipython-input-1786880988.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  gen_file = 'gpt2_gentext_{:%Y%m%d_%H%M%S}.txt'.format(datetime.utcnow())


In [ ]:
# may have to run twice to get file to download
files.download(gen_file)

## Generate Text From The Pretrained Model

If you want to generate text from the pretrained model, not a finetuned model, pass `model_name` to `gpt2.load_gpt2()` and `gpt2.generate()`.

This is currently the only way to generate text from the 774M or 1558M models with this notebook.

In [ ]:
model_name = "774M"

gpt2.download_gpt2(model_name=model_name)

Fetching checkpoint: 1.05Mit [00:00, 6.57Git/s]                                                     
Fetching encoder.json: 1.05Mit [00:00, 2.02Mit/s]
Fetching hparams.json: 1.05Mit [00:00, 7.20Git/s]                                                   
Fetching model.ckpt.data-00000-of-00001:  73%|███████████▌    | 2.25G/3.10G [03:21<01:16, 11.1Mit/s]

In [ ]:
sess = gpt2.start_tf_sess()

gpt2.load_gpt2(sess, model_name=model_name)

In [ ]:
gpt2.generate(sess,
              model_name=model_name,
              prefix="Capital of France is?",
              length=100,
              temperature=0.7,
              top_p=0.9,
              nsamples=5,
              batch_size=5
              )

# Etcetera

If the notebook has errors (e.g. GPU Sync Fail), force-kill the Colaboratory virtual machine and restart it with the command below:

In [ ]:
!kill -9 -1

# LICENSE

MIT License

Copyright (c) 2019 Max Woolf

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.